# 1.15) Defensive programming and packaging

Analysis code earns trust by failing loudly on bad input and by being tested. This subchapter separates the two tools that look similar but are not — `assert` for internal invariants and exceptions for bad input — then covers validating physical preconditions, logging instead of printing, and writing a test suite with pytest. It ends by promoting reusable code into a packaged `src/` layout, and with a generated-code bug that is invisible until the code runs in optimised mode.

:::{admonition} Learning objectives
:class: tip
- Use assert to check invariants that should hold if the code is correct.
- Signal and handle bad input with exceptions: raise, custom exception types, and try/except/else/finally.
- Validate physical preconditions explicitly.
- Prefer logging to print for messages that carry a severity level.
- Write and run a pytest suite with plain tests, a fixture, and parametrisation.
- Understand the src/ layout and lockfile that uv scaffolds for a package.
:::

## assert for invariants

An `assert` documents and checks a condition that should always be true *if the code is correct*. It is a statement about the program's own logic, not about external input — and, crucially, it is removed when Python runs with the `-O` flag.

In [1]:
def normalise(weights: list[float]) -> list[float]:
    total = sum(weights)
    result = [w / total for w in weights]
    # invariant: normalised weights sum to 1 (a check on our own arithmetic)
    assert abs(sum(result) - 1.0) < 1e-9, "normalisation failed"
    return result

print(normalise([1.0, 3.0]))

[0.25, 0.75]


## Exceptions: raise, custom types, and try/except/else/finally

An exception signals a runtime problem. `raise` triggers one; a custom exception subclass names a specific failure; `try/except/else/finally` handles it — `else` runs when no exception occurred, `finally` always runs.

In [2]:
class PhysicalRangeError(ValueError):
    # a named failure mode, more specific than a bare ValueError
    pass

def to_kelvin(temp_celsius: float) -> float:
    if temp_celsius < -273.15:
        raise PhysicalRangeError(f"{temp_celsius} °C is below absolute zero")
    return temp_celsius + 273.15

for value in [25.0, -300.0]:
    try:
        kelvin = to_kelvin(value)
    except PhysicalRangeError as err:
        print("rejected:", err)
    else:
        print("ok:", round(kelvin, 2), "K")   # runs only when no exception was raised
    finally:
        print("checked", value)               # always runs

ok: 298.15 K
checked 25.0
rejected: -300.0 °C is below absolute zero
checked -300.0


## Validating physical preconditions

External input — a file, a user value, a network response — must be validated with exceptions, not asserts, because it can be wrong even when the code is correct.

In [3]:
def validate_observation(temp_celsius: float, discharge_m3s: float) -> None:
    if temp_celsius < -273.15:
        raise PhysicalRangeError("temperature below absolute zero")
    if discharge_m3s < 0.0:
        raise ValueError("discharge cannot be negative")

validate_observation(18.0, 45.0)          # valid input passes silently
try:
    validate_observation(18.0, -5.0)
except ValueError as err:
    print("caught:", err)

caught: discharge cannot be negative


## Logging over print

`print` writes unconditionally to stdout. `logging` attaches a severity level to each message, so the same code can be verbose while debugging and quiet in production, and can route messages to files or services without edits.

In [4]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s",
                    stream=sys.stdout, force=True)
logger = logging.getLogger("mlees")

logger.debug("suppressed: below the INFO threshold")
logger.info("loaded 45 readings")
logger.warning("3 temperature values missing")

INFO: loaded 45 readings


:::{admonition} Computational-thinking fundamental: assert for bugs, exceptions for bad input
:class: important
The two mechanisms answer different questions. `assert` asks "is my code internally consistent?" — a violation is a programmer error, and the check may be compiled away in production. An exception asks "is this input acceptable?" — a violation is an expected runtime condition that must always be caught, whatever flags Python runs under. Confusing the two is a real defect: validate external input with exceptions, and reserve assert for invariants that only a bug could break.
:::

## Testing with pytest

pytest discovers functions named `test_*`, runs them, and reports failures with readable assertion output. A *fixture* supplies reusable setup; *parametrisation* runs one test over many cases. Here we write a small module and its test file to disk, then run the suite.

In [5]:
from pathlib import Path

module_src = """
def to_kelvin(temp_celsius):
    if temp_celsius < -273.15:
        raise ValueError("below absolute zero")
    return temp_celsius + 273.15
"""
Path("thermo.py").write_text(module_src, encoding="utf-8")

test_src = """
import pytest
from thermo import to_kelvin

def test_freezing_point():
    assert to_kelvin(0.0) == 273.15

@pytest.fixture
def boiling_celsius():
    return 100.0

def test_with_fixture(boiling_celsius):
    assert to_kelvin(boiling_celsius) == 373.15

@pytest.mark.parametrize("celsius, kelvin", [(0.0, 273.15), (-273.15, 0.0), (25.0, 298.15)])
def test_conversions(celsius, kelvin):
    assert to_kelvin(celsius) == pytest.approx(kelvin)

def test_below_absolute_zero_raises():
    with pytest.raises(ValueError):
        to_kelvin(-300.0)
"""
Path("test_thermo.py").write_text(test_src, encoding="utf-8")
print("wrote thermo.py and test_thermo.py")

wrote thermo.py and test_thermo.py


In [6]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_thermo.py", "-q", "--color=no"],
    capture_output=True, text=True,
)
print(result.stdout.strip())

......                                                                   [100%]
6 passed in 0.02s


## Packaging with uv: the src layout

For code you will reuse or share, promote it from a notebook to an installable package. `uv init --lib` scaffolds a `src/` layout with a `pyproject.toml`, and `uv.lock` records the exact dependency versions.

```bash
uv init --lib mypackage       # create a src/ layout with pyproject.toml
cd mypackage
uv add numpy                  # add a dependency, updating uv.lock
uv run pytest                 # run the tests in the locked environment
```

Placing code under `src/mypackage/` means tests import the *installed* package, not stray files in the working directory, so packaging mistakes surface immediately. `uv.lock` pins the full dependency graph, making the environment reproducible on any machine.

## When generated code lies: assert as input validation

Asked to reject impossible temperatures, an assistant writes `assert temp >= -273.15`. It works when you test it — and silently vanishes under `python -O`, which strips assertions, so invalid data sails through in exactly the optimised runs used in production.

In [7]:
import subprocess
import sys

# validation written with assert (as generated)
program = "temp = -300.0\nassert temp >= -273.15, 'below absolute zero'\nprint('accepted:', temp)"

normal = subprocess.run([sys.executable, "-c", program], capture_output=True, text=True)
optimised = subprocess.run([sys.executable, "-O", "-c", program], capture_output=True, text=True)

print("normal python -> exit", normal.returncode, "|", normal.stderr.strip().splitlines()[-1])
print("python -O     -> exit", optimised.returncode, "|", optimised.stdout.strip())

normal python -> exit 1 | AssertionError: below absolute zero
python -O     -> exit 0 | accepted: -300.0


:::{admonition} Diagnosis: assert is compiled out under -O
:class: warning
Under `python -O` the assertion is removed entirely, so the below-absolute-zero value is accepted with no error — a silent failure that appears only in optimised runs. assert is for internal invariants, never for validating input. The fix is an explicit `raise`, which executes regardless of optimisation flags.
:::

In [8]:
# validation written with raise: fires in every mode
program = ("temp = -300.0\n"
           "if temp < -273.15:\n"
           "    raise ValueError('below absolute zero')\n"
           "print('accepted:', temp)")

for flags, label in [([], "normal python"), (["-O"], "python -O    ")]:
    run = subprocess.run([sys.executable, *flags, "-c", program], capture_output=True, text=True)
    tail = run.stderr.strip().splitlines()[-1] if run.returncode else run.stdout.strip()
    print(label, "-> exit", run.returncode, "|", tail)

normal python -> exit 1 | ValueError: below absolute zero
python -O     -> exit 1 | ValueError: below absolute zero


:::{admonition} Going deeper: Ruff for linting and formatting
:class: seealso dropdown
[Ruff](https://docs.astral.sh/ruff/) is a fast linter and formatter that replaces flake8, isort, and black.

```bash
uv tool install ruff
ruff check .        # lint
ruff format .       # format
```

Run it in an editor and in CI so style and simple errors never reach review.
:::

:::{admonition} Going deeper: static type checking
:class: seealso dropdown
A type checker (mypy, or Astral's newer `ty`) verifies the annotations you already write, catching a class of bugs before runtime.

```bash
uvx mypy src/       # check the package against its type hints
```

Type hints plus a checker turn "this function expects a float" from a comment into an enforced contract.
:::

:::{admonition} Going deeper: pre-commit and continuous integration
:class: seealso dropdown
`pre-commit` runs checks before each commit; a CI workflow re-runs them on every push so nothing untested is merged.

```yaml
# .github/workflows/ci.yml
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v5
      - run: uv run pytest
```

Local hooks give fast feedback; CI guarantees the checks actually ran.
:::

:::{admonition} Going deeper: semantic versioning and coverage
:class: seealso dropdown
Semantic versioning communicates the nature of a change through the version number `MAJOR.MINOR.PATCH`: bump MAJOR for breaking changes, MINOR for new features, PATCH for fixes. Coverage measures how much code the tests exercise.

```bash
uv run pytest --cov=mypackage      # report line coverage
```

Coverage is a guide, not a target: high coverage of meaningless tests proves nothing.
:::

:::{admonition} Takeaways
:class: danger
- `assert` checks internal invariants and is stripped by `python -O`; never use it to validate external input.
- Raise exceptions (a custom subclass when it clarifies intent) for bad input; handle with try/except/else/finally.
- Validate physical preconditions explicitly, and fail early and loudly.
- Prefer `logging` to `print`: messages gain a severity level and can be filtered or redirected.
- Test with pytest — plain tests, fixtures for setup, parametrisation for many cases — and run the suite in CI.
- Promote reusable code to a `src/` package with `uv init --lib`; pin dependencies with `uv.lock`.
:::

## Resources

- [pytest documentation](https://docs.pytest.org/en/stable/) — writing tests, fixtures, parametrisation, and assertions.
- [uv documentation](https://docs.astral.sh/uv/) — project creation, the src layout, dependency management, and the lockfile.
- [Ruff documentation](https://docs.astral.sh/ruff/) — the linter and formatter referenced in the Going-deeper boxes.
:::